#Overpass Data Fetch

https://overpass-turbo.eu/

```
Area: Twin Cities metro
Type: Food & Drink · Shopping · Leisure · Tourism · Health · Education · Public Services
```





**Fetch and Concat**

In [2]:
pip install requests

In [3]:
import requests
import time
import os
import csv
import sys

# ── Configuration ──────────────────────────────────────────────────────────────
OUTPUT_DIR = "."
BOUNDING_BOX = "44.7,-93.7,45.2,-92.9"   # Twin Cities metro
TIMEOUT = 120
RETRY_ATTEMPTS = 3
RETRY_DELAY = 10

SERVERS = [
    "https://overpass-api.de/api/interpreter",
    "https://lz4.overpass-api.de/api/interpreter",
    "https://z.overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
]

# ── Query definitions ──────────────────────────────────────────────────────────
# Each query is a simple single nwr[...](bbox); block — no union brackets needed
def make_query(filter_clause):
    return (
        f'[out:csv(::"id",::"lat",::"lon",::"type",name,amenity,shop,leisure,'
        f'"natural",waterway,tourism,cuisine,brand,operator,'
        f'"addr:housenumber","addr:street","addr:city","addr:postcode","addr:state",'
        f'phone,website,"opening_hours",wheelchair,fee;true;",")][timeout:{TIMEOUT}];\n'
        f'{filter_clause}({BOUNDING_BOX});\n'
        f'out center;'
    )

QUERIES = [
    {
        "name": "food_drink",
        "label": "Food & Drink",
        "query": make_query(
            'nwr["amenity"~"restaurant|cafe|fast_food|bar|pub|ice_cream|food_court|bubble_tea|biergarten|bbq"]'
        ),
    },
    {
        "name": "edu_health_finance",
        "label": "Education / Health / Finance",
        "query": make_query(
            'nwr["amenity"~"school|university|college|kindergarten|library|hospital|clinic|pharmacy|dentist|doctors|veterinary|nursing_home|bank|atm|bureau_de_change"]'
        ),
    },
    {
        "name": "transport_public",
        "label": "Transport / Entertainment / Public Services",
        "query": make_query(
            'nwr["amenity"~"parking|fuel|bus_station|car_rental|car_wash|charging_station|cinema|theatre|nightclub|casino|community_centre|arts_centre|place_of_worship|post_office|police|fire_station|townhall|courthouse|recycling|toilets"]'
        ),
    },
    {
        "name": "shops",
        "label": "Shops",
        "query": make_query(
            'nwr["shop"~"supermarket|mall|department_store|clothes|second_hand|convenience|electronics|books|bakery|butcher|hardware|florist|alcohol|furniture|shoes|jewelry|optician|toys|sports|pet|garden_centre|car|bicycle|music|gift|cosmetics|mobile_phone|stationery|art|antiques|deli|seafood|health_food|dry_cleaning|laundry"]'
        ),
    },
    # ── Split leisure and nature into two separate queries ──
    {
        "name": "leisure",
        "label": "Leisure",
        "query": make_query(
            'nwr["leisure"~"park|playground|sports_centre|fitness_centre|swimming_pool|golf_course|marina|nature_reserve|stadium|ice_rink|bowling_alley|dog_park|garden|amusement_arcade|escape_game"]'
        ),
    },
    {
        "name": "nature",
        "label": "Nature / Waterways",
        "query": make_query(
            'nwr["natural"~"water|beach|wetland|wood|grassland|river"]'
        ),
    },
    {
        "name": "tourism",
        "label": "Tourism",
        "query": make_query(
            'nwr["tourism"~"museum|hotel|attraction|viewpoint|zoo|camp_site|hostel|motel|guest_house|theme_park|artwork|gallery|aquarium|information"]'
        ),
    },
]

# ── Core fetch function ────────────────────────────────────────────────────────
def fetch_query(query_str, label):
    for server in SERVERS:
        for attempt in range(1, RETRY_ATTEMPTS + 1):
            try:
                print(f"  [{label}] Trying {server} (attempt {attempt}/{RETRY_ATTEMPTS})...")
                resp = requests.post(
                    server,
                    data={"data": query_str},
                    timeout=TIMEOUT + 10,
                    headers={"User-Agent": "SEIS732-Student-Project/1.0"},
                )
                if resp.status_code == 200 and resp.text.strip():
                    lines = resp.text.strip().splitlines()
                    if len(lines) >= 2:
                        print(f"  [{label}]  Got {len(lines)-1} rows")
                        return resp.text
                    else:
                        print(f"  [{label}]   Empty response (0 data rows)")
                else:
                    print(f"  [{label}]   HTTP {resp.status_code} or empty body")

            except requests.exceptions.Timeout:
                print(f"  [{label}]   Timeout on attempt {attempt}")
            except requests.exceptions.ConnectionError as e:
                print(f"  [{label}] Connection error: {e}")
            except Exception as e:
                print(f"  [{label}]  Unexpected error: {e}")

            if attempt < RETRY_ATTEMPTS:
                print(f"  [{label}] Waiting {RETRY_DELAY}s before retry...")
                time.sleep(RETRY_DELAY)

        print(f"  [{label}]  All attempts failed for {server}, trying next server...")
        time.sleep(5)

    return None

# ── Save individual CSV ────────────────────────────────────────────────────────
def save_csv(filename, csv_text):
    path = os.path.join(OUTPUT_DIR, filename)
    with open(path, "w", encoding="utf-8", newline="") as f:
        f.write(csv_text)
    print(f"   Saved: {path}")
    return path

# ── Merge all CSVs into master ─────────────────────────────────────────────────
def merge_csvs(file_paths, output_file):
    print(f"\n Merging {len(file_paths)} files into {output_file}...")
    seen_ids = set()
    total_rows = 0

    with open(output_file, "w", encoding="utf-8", newline="") as out_f:
        writer = None

        for path in file_paths:
            if not os.path.exists(path):
                print(f"    Skipping missing file: {path}")
                continue

            with open(path, "r", encoding="utf-8") as in_f:
                reader = csv.DictReader(in_f)
                if writer is None:
                    writer = csv.DictWriter(out_f, fieldnames=reader.fieldnames)
                    writer.writeheader()
                for row in reader:
                    row_id = row.get("@id", "")
                    if row_id and row_id not in seen_ids:
                        seen_ids.add(row_id)
                        writer.writerow(row)
                        total_rows += 1

    print(f"   Merged {total_rows} unique rows → {output_file}")
    return total_rows

# ── Main ───────────────────────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("  Twin Cities POI Scraper v2 — Overpass API")
    print("=" * 60)
    print(f"  Bounding box : {BOUNDING_BOX}")
    print(f"  Output dir   : {os.path.abspath(OUTPUT_DIR)}")
    print(f"  Total queries: {len(QUERIES)}")
    print()

    saved_files = []

    for q in QUERIES:
        print(f"\n Fetching: {q['label']}")
        csv_text = fetch_query(q["query"], q["label"])

        if csv_text:
            filename = f"mn_poi_{q['name']}.csv"
            path = save_csv(filename, csv_text)
            saved_files.append(path)
        else:
            print(f"   FAILED to fetch {q['label']} — skipping")

        print(f"   Waiting 15s before next query...")
        time.sleep(15)

    if saved_files:
        master_path = os.path.join(OUTPUT_DIR, "mn_poi_master.csv")
        total = merge_csvs(saved_files, master_path)
        print(f"\n{'='*60}")
        print(f"   Done! Master file : {master_path}")
        print(f"   Total unique POIs : {total}")
        print(f"{'='*60}")
    else:
        print("\n No files downloaded. Check your internet connection.")
        sys.exit(1)

if __name__ == "__main__":
    main()

  Twin Cities POI Scraper v2 — Overpass API
  Bounding box : 44.7,-93.7,45.2,-92.9
  Output dir   : /content
  Total queries: 7


 Fetching: Food & Drink
  [Food & Drink] Trying https://overpass-api.de/api/interpreter (attempt 1/3)...
  [Food & Drink]  Got 4783 rows
   Saved: ./mn_poi_food_drink.csv
   Waiting 15s before next query...

 Fetching: Education / Health / Finance
  [Education / Health / Finance] Trying https://overpass-api.de/api/interpreter (attempt 1/3)...
  [Education / Health / Finance]  Got 2336 rows
   Saved: ./mn_poi_edu_health_finance.csv
   Waiting 15s before next query...

 Fetching: Transport / Entertainment / Public Services
  [Transport / Entertainment / Public Services] Trying https://overpass-api.de/api/interpreter (attempt 1/3)...
  [Transport / Entertainment / Public Services]  Got 35857 rows
   Saved: ./mn_poi_transport_public.csv
   Waiting 15s before next query...

 Fetching: Shops
  [Shops] Trying https://overpass-api.de/api/interpreter (attempt 1/3)...

**Data Exploring**

In [4]:
import csv

filepath = "./mn_poi_master.csv"

with open(filepath, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    print("Columns:", reader.fieldnames)
    print()
    for i, row in enumerate(reader):
        print(f"Row {i+1}:", dict(row))
        if i >= 4:   # show top 5 rows
            break

# Also print total row count
with open(filepath, "r", encoding="utf-8") as f:
    total = sum(1 for _ in f) - 1  # subtract header
print(f"\nTotal rows: {total}")

Columns: ['@id', '@lat', '@lon', '@type', 'name', 'amenity', 'shop', 'leisure', 'natural', 'waterway', 'tourism', 'cuisine', 'brand', 'operator', 'addr:housenumber', 'addr:street', 'addr:city', 'addr:postcode', 'addr:state', 'phone', 'website', 'opening_hours', 'wheelchair', 'fee']

Row 1: {'@id': '319434041', '@lat': '44.9857583', '@lon': '-93.2762636', '@type': 'node', 'name': 'Corner Coffee', 'amenity': 'cafe', 'shop': '', 'leisure': '', 'natural': '', 'waterway': '', 'tourism': '', 'cuisine': 'coffee_shop', 'brand': '', 'operator': '', 'addr:housenumber': '514', 'addr:street': '3rd Street North', 'addr:city': 'Minneapolis', 'addr:postcode': '55401', 'addr:state': 'MN', 'phone': '+1-612-338-2002', 'website': 'https://corner.coffee', 'opening_hours': 'Mo-Fr 07:00-17:00; Sa 08:00-17:00', 'wheelchair': '', 'fee': ''}
Row 2: {'@id': '415414878', '@lat': '44.7324247', '@lon': '-93.2235998', '@type': 'node', 'name': "Denny's", 'amenity': 'restaurant', 'shop': '', 'leisure': '', 'natural':

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import shutil

source_path = "./mn_poi_master.csv"
dest_path = "/content/drive/MyDrive/ConversationalAI/mn_poi_master.csv"

shutil.copy(source_path, dest_path)

print("File saved to Google Drive!")

File saved to Google Drive!


In [7]:
from google.colab import files

files.download("./mn_poi_master.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
import pandas as pd

filepath = "./mn_poi_master.csv"

df = pd.read_csv(filepath)

# Show basic info
print("Columns:", df.columns.tolist())
print("\nShape:", df.shape)

Columns: ['@id', '@lat', '@lon', '@type', 'name', 'amenity', 'shop', 'leisure', 'natural', 'waterway', 'tourism', 'cuisine', 'brand', 'operator', 'addr:housenumber', 'addr:street', 'addr:city', 'addr:postcode', 'addr:state', 'phone', 'website', 'opening_hours', 'wheelchair', 'fee']

Shape: (81674, 24)


/tmp/ipykernel_4756/2586231734.py:5: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath)


In [9]:
df.head()

,@id,@lat,@lon,@type,name,amenity,shop,leisure,natural,waterway,...,addr:housenumber,addr:street,addr:city,addr:postcode,addr:state,phone,website,opening_hours,wheelchair,fee
0,319434041,44.985758,-93.276264,node,Corner Coffee,cafe,NaN,NaN,NaN,NaN,...,514,3rd Street North,Minneapolis,55401,MN,+1-612-338-2002,https://corner.coffee,Mo-Fr 07:00-17:00; Sa 08:00-17:00,NaN,NaN
1,415414878,44.732425,-93.223600,node,Denny's,restaurant,NaN,NaN,NaN,NaN,...,7805,150th Street West,Apple Valley,55124,MN,+1 952-432-2022,https://locations.dennys.com/MN/APPLE-VALLEY/2...,"Mo,Tu,Su 05:00-24:00, We-Sa 05:00-02:00",yes,NaN
2,415414880,44.732385,-93.218746,node,Raising Cane's,fast_food,NaN,NaN,NaN,NaN,...,7501,150th Street West,Apple Valley,55124,MN,+1-952-432-8700,https://locations.raisingcanes.com/mn/apple-va...,"Su-Th 10:00-24:00, Fr,Sa 10:00-02:00",yes,NaN
3,415414886,44.723856,-93.216915,node,Red Robin,restaurant,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,430830897,44.916080,-93.210885,node,Sea Salt Eatery,fast_food,NaN,NaN,NaN,NaN,...,4801,South Minnehaha Avenue,Minneapolis,55417,MN,+1-612-721-8990,https://www.seasaltmpls.com/,NaN,NaN,NaN


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81674 entries, 0 to 81673
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   @id               81674 non-null  int64  
 1   @lat              81674 non-null  float64
 2   @lon              81674 non-null  float64
 3   @type             81674 non-null  object 
 4   name              17281 non-null  object 
 5   amenity           42988 non-null  object 
 6   shop              3929 non-null   object 
 7   leisure           10961 non-null  object 
 8   natural           22375 non-null  object 
 9   waterway          0 non-null      float64
 10  tourism           1537 non-null   object 
 11  cuisine           3028 non-null   object 
 12  brand             4648 non-null   object 
 13  operator          2431 non-null   object 
 14  addr:housenumber  8763 non-null   object 
 15  addr:street       8781 non-null   object 
 16  addr:city         7591 non-null   object

In [11]:
for col in df.columns:
    print("="*50)
    print(f"Column: {col}")

    print("Type:", df[col].dtype)
    print("Nulls:", df[col].isna().sum())
    print("Unique values:", df[col].nunique())

    # Show sample values
    print("\nSample values:")
    print(df[col].dropna().head(5))

    # Numeric vs categorical
    if pd.api.types.is_numeric_dtype(df[col]):
        print("\nStats:")
        print(df[col].describe())
    else:
        print("\nTop values:")
        print(df[col].value_counts().head(5))

Column: @id
Type: int64
Nulls: 0
Unique values: 81674

Sample values:
0    319434041
1    415414878
2    415414880
3    415414886
4    430830897
Name: @id, dtype: int64

Stats:
count    8.167400e+04
mean     2.104992e+09
std      3.429876e+09
min      3.573600e+04
25%      4.351809e+08
50%      9.929417e+08
75%      1.333831e+09
max      1.376107e+10
Name: @id, dtype: float64
Column: @lat
Type: float64
Nulls: 0
Unique values: 79199

Sample values:
0    44.985758
1    44.732425
2    44.732385
3    44.723856
4    44.916080
Name: @lat, dtype: float64

Stats:
count    81674.000000
mean        44.943994
std          0.116569
min         44.559067
25%         44.860004
50%         44.947437
75%         45.015289
max         45.212001
Name: @lat, dtype: float64
Column: @lon
Type: float64
Nulls: 0
Unique values: 80061

Sample values:
0   -93.276264
1   -93.223600
2   -93.218746
3   -93.216915
4   -93.210885
Name: @lon, dtype: float64

Stats:
count    81674.000000
mean       -93.255306
std     

In [12]:
summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.values,
    "nulls": df.isna().sum().values,
    "null_pct": (df.isna().mean()*100).round(2).values,
    "unique": df.nunique().values
})

summary

,column,dtype,nulls,null_pct,unique
0,@id,int64,0,0.00,81674
1,@lat,float64,0,0.00,79199
2,@lon,float64,0,0.00,80061
3,@type,object,0,0.00,3
4,name,object,64393,78.84,12168
5,amenity,object,38686,47.37,69
6,shop,object,77745,95.19,63
7,leisure,object,70713,86.58,21
8,natural,object,59299,72.60,6
9,waterway,float64,81674,100.00,0


In [13]:
df[df['name'].str.contains("lu's|lu sandwich", case=False, na=False)]

,@id,@lat,@lon,@type,name,amenity,shop,leisure,natural,waterway,...,addr:housenumber,addr:street,addr:city,addr:postcode,addr:state,phone,website,opening_hours,wheelchair,fee
258,922237260,44.974674,-93.194521,node,Lulu's Salsa,restaurant,NaN,NaN,NaN,NaN,...,2233,Energy Park Drive,Saint Paul,55108,MN,+1 651-202-3450,NaN,08:00-22:00,NaN,NaN
1004,4619562333,44.954727,-93.278049,node,Lu's Sandwiches,restaurant,NaN,NaN,NaN,NaN,...,2624,Nicollet Avenue South,Minneapolis,55408,MN,+1 612-587-2694,https://www.lussandwiches.com,Mo-Sa 10:00-20:00; Su 10:00-18:00,yes,NaN
1841,9127859059,44.989950,-93.252665,node,Lu's Sandwiches,restaurant,NaN,NaN,NaN,NaN,...,10,Northeast 6th Street,NaN,55413,NaN,NaN,https://lusandwiches.com,Mo-Sa 11:00-20:00; Su 11:00-19:00,NaN,NaN
3873,177770113,44.981205,-93.177617,way,Lulu's Public House,restaurant,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Data Preprocessing**

In [20]:
"""
POI Data Preprocessing (Initial Clean)
=====================================
Logic:
  - Read raw POI CSV input file
  - Drop unnecessary columns:
      • @type, waterway, operator
  - Remove invalid records:
      • Drop rows where NAME is blank or missing
  - Preserve all other fields as-is (no transformation)
  - Write cleaned dataset to new CSV file
  - Track processing stats:
      • total input rows
      • rows dropped (missing name)
      • total output rows
  - Output file contains only relevant columns with valid POIs
"""

import csv
import os

INPUT_FILE  = "./mn_poi_master.csv"
OUTPUT_FILE = "./mn_poi_clean.csv"

DROP_COLUMNS = {"@type", "waterway", "operator"}

def clean(value):
    return value.strip() if value else ""

def main():
    print("=" * 50)
    print("  POI Data Preprocessor")
    print("=" * 50)

    if not os.path.exists(INPUT_FILE):
        print(f" File not found: {INPUT_FILE}")
        return

    total_in   = 0
    total_out  = 0
    dropped_name = 0

    with open(INPUT_FILE, "r", encoding="utf-8") as in_f, \
         open(OUTPUT_FILE, "w", encoding="utf-8", newline="") as out_f:

        reader = csv.DictReader(in_f)

        # Build output fieldnames — remove dropped columns
        out_fields = [f for f in reader.fieldnames if f not in DROP_COLUMNS]
        writer = csv.DictWriter(out_f, fieldnames=out_fields)
        writer.writeheader()

        for row in reader:
            total_in += 1

            # Drop rows with blank name
            if not clean(row.get("name", "")):
                dropped_name += 1
                continue

            # Write row with dropped columns removed
            out_row = {k: v for k, v in row.items() if k not in DROP_COLUMNS}
            writer.writerow(out_row)
            total_out += 1

    print(f"  Input rows      : {total_in:,}")
    print(f"  Dropped (no name): {dropped_name:,}")
    print(f"  Output rows     : {total_out:,}")
    print(f"\n   Saved: {OUTPUT_FILE}")
    print("=" * 50)

if __name__ == "__main__":
    main()

  POI Data Preprocessor
  Input rows      : 81,674
  Dropped (no name): 64,393
  Output rows     : 17,281

   Saved: ./mn_poi_clean.csv


In [19]:
"""
POI Data Cleaning & Categorization
=================================
Logic:
  - Rename raw OSM columns → standardized schema (POI_ID, LATITUDE, etc.)
  - Normalize key fields:
      • PHONE → +1-XXX-XXX-XXXX format
      • POSTCODE → remove ZIP+4 suffix
      • STATE → enforce "MN"
  - CATEGORY assignment (priority order):
      • AMENITY → SHOP → LEISURE → NATURAL → TOURISM → else "Other"
  - SUBCATEGORY derivation:
      • SUB_CATEGORY_1 = first non-empty among (AMENITY, SHOP, LEISURE, NATURAL, TOURISM)
      • SUB_CATEGORY_2 = second non-empty value
      • If only AMENITY exists → SUB_CATEGORY_2 = CUISINE (first value)
  - CUISINE preserved as-is (no expansion)
  - Drop original classification columns after consolidation:
      • AMENITY, SHOP, LEISURE, NATURAL, TOURISM, BRAND
  - Output fixed schema with ordered columns
  - Generate summary:
      • row/column counts
      • category distribution
      • sample preview
"""

import csv, re, os
from collections import Counter

INPUT_FILE  = "./mn_poi_clean.csv"
OUTPUT_FILE = "./mn_poi_final.csv"

RENAME = {
    "@id":"POI_ID","@lat":"LATITUDE","@lon":"LONGITUDE","name":"NAME",
    "amenity":"AMENITY","shop":"SHOP","leisure":"LEISURE","natural":"NATURAL",
    "tourism":"TOURISM","cuisine":"CUISINE","brand":"BRAND",
    "addr:housenumber":"HOUSE_NUMBER","addr:street":"STREET","addr:city":"CITY",
    "addr:postcode":"POSTCODE","addr:state":"STATE","phone":"PHONE",
    "website":"WEBSITE","opening_hours":"OPENING_HOURS",
    "wheelchair":"WHEELCHAIR","fee":"FEE",
}

# Columns to drop after deriving SUB_CATEGORY_1 and SUB_CATEGORY_2
DROP_AFTER = {"AMENITY","SHOP","LEISURE","NATURAL","TOURISM","BRAND"}

AMENITY_CAT = {
    "restaurant":"Food & Drink","cafe":"Food & Drink","fast_food":"Food & Drink",
    "bar":"Food & Drink","pub":"Food & Drink","ice_cream":"Food & Drink",
    "food_court":"Food & Drink","bubble_tea":"Food & Drink","biergarten":"Food & Drink","bbq":"Food & Drink",
    "school":"Education","university":"Education","college":"Education",
    "kindergarten":"Education","library":"Education","public_bookcase":"Education",
    "hospital":"Health","clinic":"Health","pharmacy":"Health","dentist":"Health",
    "doctors":"Health","veterinary":"Health","nursing_home":"Health",
    "bank":"Finance","atm":"Finance","bureau_de_change":"Finance",
    "parking":"Transport","fuel":"Transport","bus_station":"Transport",
    "car_rental":"Transport","car_wash":"Transport","charging_station":"Transport",
    "bicycle_rental":"Transport","ferry_terminal":"Transport",
    "cinema":"Entertainment","theatre":"Entertainment","nightclub":"Entertainment",
    "casino":"Entertainment","arts_centre":"Entertainment","community_centre":"Entertainment",
    "place_of_worship":"Public Services","post_office":"Public Services",
    "police":"Public Services","fire_station":"Public Services",
    "townhall":"Public Services","courthouse":"Public Services",
    "recycling":"Public Services","toilets":"Public Services",
}
SHOP_CAT = {
    "supermarket":"Grocery & Food","convenience":"Grocery & Food","bakery":"Grocery & Food",
    "butcher":"Grocery & Food","deli":"Grocery & Food","seafood":"Grocery & Food",
    "health_food":"Grocery & Food","alcohol":"Grocery & Food","pastry":"Food & Drink","coffee":"Food & Drink",
    "clothes":"Shopping","second_hand":"Shopping","electronics":"Shopping","books":"Shopping",
    "hardware":"Shopping","florist":"Shopping","furniture":"Shopping","shoes":"Shopping",
    "jewelry":"Shopping","optician":"Shopping","toys":"Shopping","sports":"Shopping",
    "pet":"Shopping","garden_centre":"Shopping","car":"Shopping","bicycle":"Shopping",
    "music":"Shopping","gift":"Shopping","cosmetics":"Shopping","mobile_phone":"Shopping",
    "stationery":"Shopping","art":"Shopping","antiques":"Shopping",
    "mall":"Shopping","department_store":"Shopping",
    "dry_cleaning":"Services","laundry":"Services",
}
LEISURE_CAT = {v:"Leisure" for v in ["park","playground","sports_centre","fitness_centre",
    "swimming_pool","golf_course","marina","nature_reserve","stadium","ice_rink",
    "bowling_alley","dog_park","garden","amusement_arcade","escape_game","music_venue"]}
NATURAL_CAT = {v:"Nature" for v in ["water","beach","wetland","wood","grassland","river"]}
TOURISM_CAT = {v:"Tourism" for v in ["museum","hotel","attraction","viewpoint","zoo",
    "camp_site","hostel","motel","guest_house","theme_park","artwork","gallery","aquarium","information"]}

def get_category(row):
    a,s,l,n,t = row.get("AMENITY","").strip(),row.get("SHOP","").strip(),row.get("LEISURE","").strip(),row.get("NATURAL","").strip(),row.get("TOURISM","").strip()
    if a in AMENITY_CAT: return AMENITY_CAT[a]
    if s in SHOP_CAT:    return SHOP_CAT[s]
    if l in LEISURE_CAT: return LEISURE_CAT[l]
    if n in NATURAL_CAT: return NATURAL_CAT[n]
    if t in TOURISM_CAT: return TOURISM_CAT[t]
    return "Other"

def get_subs(row):
    cols = [("AMENITY",row.get("AMENITY","").strip()),("SHOP",row.get("SHOP","").strip()),
            ("LEISURE",row.get("LEISURE","").strip()),("NATURAL",row.get("NATURAL","").strip()),
            ("TOURISM",row.get("TOURISM","").strip())]
    filled = [(c,v) for c,v in cols if v]
    sub1 = filled[0][1] if filled else ""
    sub2 = filled[1][1] if len(filled)>=2 else ""
    if len(filled)==1 and filled[0][0]=="AMENITY":
        cuisine = row.get("CUISINE","").strip()
        if cuisine: sub2 = cuisine.split(";")[0].strip()
    return sub1, sub2

def norm_phone(p):
    if not p.strip(): return ""
    d = re.sub(r"\D","",p)
    if len(d)==11 and d[0]=="1": d=d[1:]
    return f"+1-{d[0:3]}-{d[3:6]}-{d[6:10]}" if len(d)==10 else p.strip()

def clean_zip(z):
    return z.strip().split("-")[0] if z.strip() else ""

def norm_state(s):
    return "MN" if s.strip().upper() in ("MN","MINNESOTA","") else s.strip().upper()

with open(INPUT_FILE,"r",encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

renamed = []
for row in rows:
    nr = {RENAME.get(k,k.upper()): v for k,v in row.items()}
    nr["STATE"]           = norm_state(nr.get("STATE",""))
    nr["PHONE"]           = norm_phone(nr.get("PHONE",""))
    nr["POSTCODE"]        = clean_zip(nr.get("POSTCODE",""))
    nr["CATEGORY"]        = get_category(nr)
    nr["SUB_CATEGORY_1"], nr["SUB_CATEGORY_2"] = get_subs(nr)
    # Drop consolidated source columns
    for col in DROP_AFTER:
        nr.pop(col, None)
    renamed.append(nr)

# Final column order — consolidated columns removed
FIELDS = [
    "POI_ID","LATITUDE","LONGITUDE","NAME",
    "CATEGORY","SUB_CATEGORY_1","SUB_CATEGORY_2",
    "CUISINE","HOUSE_NUMBER","STREET","CITY","POSTCODE","STATE",
    "PHONE","WEBSITE","OPENING_HOURS","WHEELCHAIR","FEE"
]

with open(OUTPUT_FILE,"w",encoding="utf-8",newline="") as f:
    w = csv.DictWriter(f, fieldnames=FIELDS, extrasaction="ignore")
    w.writeheader()
    w.writerows(renamed)

print(f"Rows: {len(renamed):,}  Cols: {len(FIELDS)}")
print(f"Dropped columns: {sorted(DROP_AFTER)}")
print(f"\nFinal columns: {FIELDS}")
print("\nSample rows:")
for r in renamed[:5]:
    print(f"  {r['NAME']:<30} | CAT={r['CATEGORY']:<15} | SUB1={r['SUB_CATEGORY_1']:<20} | SUB2={r['SUB_CATEGORY_2']}")

cats = Counter(r["CATEGORY"] for r in renamed)
print("\nCategory breakdown:")
for cat, cnt in sorted(cats.items(), key=lambda x:-x[1]):
    print(f"  {cat:<22} {cnt:>6,}")

Rows: 17,281  Cols: 18
Dropped columns: ['AMENITY', 'BRAND', 'LEISURE', 'NATURAL', 'SHOP', 'TOURISM']

Final columns: ['POI_ID', 'LATITUDE', 'LONGITUDE', 'NAME', 'CATEGORY', 'SUB_CATEGORY_1', 'SUB_CATEGORY_2', 'CUISINE', 'HOUSE_NUMBER', 'STREET', 'CITY', 'POSTCODE', 'STATE', 'PHONE', 'WEBSITE', 'OPENING_HOURS', 'WHEELCHAIR', 'FEE']

Sample rows:
  Corner Coffee                  | CAT=Food & Drink    | SUB1=cafe                 | SUB2=coffee_shop
  Denny's                        | CAT=Food & Drink    | SUB1=restaurant           | SUB2=american
  Raising Cane's                 | CAT=Food & Drink    | SUB1=fast_food            | SUB2=chicken
  Red Robin                      | CAT=Food & Drink    | SUB1=restaurant           | SUB2=burger
  Sea Salt Eatery                | CAT=Food & Drink    | SUB1=fast_food            | SUB2=seafood

Category breakdown:
  Food & Drink            4,032
  Leisure                 2,969
  Shopping                1,961
  Public Services         1,211
  Transpo

In [18]:
"""
Cuisine Hierarchy Expander
===========================
Logic:
  - SUB_CATEGORY_1 is fixed (never changes)
  - SUB_CATEGORY_2: if == SUB_1 → blank; else keep
  - CUISINE split by ";" → candidates for SUB_3, SUB_4, SUB_5...
  - Each cuisine part added as next SUB_CATEGORY slot IF not already in any earlier slot
  - If SUB_2 is blank but SUB_3+ has data → push up to SUB_2
  - All SUB_CATEGORY columns kept in final output
"""

import csv
import os
from collections import Counter

INPUT_FILE  = "./mn_poi_final.csv"
OUTPUT_FILE = "./mn_poi_final_v2.csv"

def expand_subcategories(row):
    sub1 = row.get("SUB_CATEGORY_1", "").strip()
    sub2 = row.get("SUB_CATEGORY_2", "").strip()
    cuisine = row.get("CUISINE", "").strip()

    # ── Rule 1: If SUB_2 == SUB_1, blank it ───────────────────────────────────
    if sub2 and sub2 == sub1:
        sub2 = ""

    # ── Rule 2: Split cuisine into parts ──────────────────────────────────────
    cuisine_parts = [c.strip() for c in cuisine.split(";") if c.strip()] if cuisine else []

    # ── Rule 3: Build sub slots starting from sub3 ────────────────────────────
    # Track what's already assigned to avoid duplicates
    assigned = set()
    if sub1: assigned.add(sub1)
    if sub2: assigned.add(sub2)

    extra_subs = []
    for part in cuisine_parts:
        if part not in assigned:
            extra_subs.append(part)
            assigned.add(part)

    # Fill sub3, sub4, sub5... from extra_subs
    sub3 = extra_subs[0] if len(extra_subs) > 0 else ""
    sub4 = extra_subs[1] if len(extra_subs) > 1 else ""
    sub5 = extra_subs[2] if len(extra_subs) > 2 else ""
    sub6 = extra_subs[3] if len(extra_subs) > 3 else ""
    sub7 = extra_subs[4] if len(extra_subs) > 4 else ""

    # ── Rule 4: Push up — if SUB_2 blank but SUB_3 has data, promote ──────────
    subs = [sub1, sub2, sub3, sub4, sub5, sub6, sub7]

    # Remove blanks in middle, compact, then pad back to 7
    compacted = [s for s in subs if s]
    # Pad to 7
    while len(compacted) < 7:
        compacted.append("")

    return {
        "SUB_CATEGORY_1": compacted[0],
        "SUB_CATEGORY_2": compacted[1],
        "SUB_CATEGORY_3": compacted[2],
        "SUB_CATEGORY_4": compacted[3],
        "SUB_CATEGORY_5": compacted[4],
        "SUB_CATEGORY_6": compacted[5],
        "SUB_CATEGORY_7": compacted[6],
    }

def main():
    print("=" * 55)
    print("  Cuisine Hierarchy Expander")
    print("=" * 55)

    if not os.path.exists(INPUT_FILE):
        print(f" File not found: {INPUT_FILE}")
        return

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    print(f"  Input rows : {len(rows):,}")

    processed = []
    stats = {
        "sub2_cleared":    0,   # SUB2 blanked because == SUB1
        "pushed_up":       0,   # SUB3 promoted to SUB2
        "sub3_filled":     0,
        "sub4_filled":     0,
        "sub5_plus":       0,
    }

    for row in rows:
        old_sub2 = row.get("SUB_CATEGORY_2","").strip()
        subs = expand_subcategories(row)

        # Track stats
        if old_sub2 and subs["SUB_CATEGORY_2"] != old_sub2 and subs["SUB_CATEGORY_2"] == "":
            stats["sub2_cleared"] += 1
        if not old_sub2 and subs["SUB_CATEGORY_2"]:
            stats["pushed_up"] += 1
        if subs["SUB_CATEGORY_3"]: stats["sub3_filled"] += 1
        if subs["SUB_CATEGORY_4"]: stats["sub4_filled"] += 1
        if subs["SUB_CATEGORY_5"] or subs["SUB_CATEGORY_6"] or subs["SUB_CATEGORY_7"]:
            stats["sub5_plus"] += 1

        # Merge back into row
        row.update(subs)
        processed.append(row)

    # Final column order
    FIELDS = [
        "POI_ID", "LATITUDE", "LONGITUDE", "NAME",
        "CATEGORY",
        "SUB_CATEGORY_1", "SUB_CATEGORY_2", "SUB_CATEGORY_3",
        "SUB_CATEGORY_4", "SUB_CATEGORY_5", "SUB_CATEGORY_6", "SUB_CATEGORY_7",
        "CUISINE",
        "HOUSE_NUMBER", "STREET", "CITY", "POSTCODE", "STATE",
        "PHONE", "WEBSITE", "OPENING_HOURS", "WHEELCHAIR", "FEE"
    ]

    with open(OUTPUT_FILE, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(processed)

    print(f"  Output rows: {len(processed):,}")
    print(f"  Output cols: {len(FIELDS)}")
    print(f"  Output file: {OUTPUT_FILE}")
    print()
    print("  === Processing stats ===")
    print(f"    SUB2 blanked (was == SUB1)     : {stats['sub2_cleared']:>5,}")
    print(f"    SUB3 promoted to SUB2 (push up): {stats['pushed_up']:>5,}")
    print(f"    Rows with SUB_CATEGORY_3       : {stats['sub3_filled']:>5,}")
    print(f"    Rows with SUB_CATEGORY_4       : {stats['sub4_filled']:>5,}")
    print(f"    Rows with SUB_CATEGORY_5+      : {stats['sub5_plus']:>5,}")
    print()

    # Show some interesting examples
    print("  === Sample multi-sub rows ===")
    examples = [r for r in processed if r.get("SUB_CATEGORY_3","")]
    for r in examples[:8]:
        subs = [r[f"SUB_CATEGORY_{i}"] for i in range(1,8) if r.get(f"SUB_CATEGORY_{i}","")]
        print(f"    {r['NAME']:<35} → {' | '.join(subs)}")

    print()
    print("  Done!")
    print("=" * 55)

if __name__ == "__main__":
    main()

  Cuisine Hierarchy Expander
  Input rows : 17,281
  Output rows: 17,281
  Output cols: 23
  Output file: ./mn_poi_final_v2.csv

  === Processing stats ===
    SUB2 blanked (was == SUB1)     :    20
    SUB3 promoted to SUB2 (push up):    11
    Rows with SUB_CATEGORY_3       :   433
    Rows with SUB_CATEGORY_4       :   122
    Rows with SUB_CATEGORY_5+      :    34

  === Sample multi-sub rows ===
    Sea Salt Eatery                     → fast_food | seafood | cajun
    Andiamo Italian Ristorante          → restaurant | italian | burger
    Ginkgo Coffeehouse                  → cafe | tea | sandwich | coffee_shop
    Panera Bread                        → fast_food | sandwich | bakery
    Cafe Latte                          → restaurant | pizza | dessert | salad
    Fogo de ChÃ£o                       → restaurant | brazilian | steak_house
    May Day Cafe                        → cafe | breakfast | coffee_shop | donut | tea
    Groveland Tap                       → pub | american | 

In [21]:
from google.colab import files

files.download("./mn_poi_final_v2.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>